# 模型静态导出通俗讲解
#### 前置大白话铺垫：PyTorch 默认是「现场做饭」动态图
原生PyTorch eager模式是边跑边算：
每输入一张图片，Python逐行执行代码，临时搭建计算流程、走判断、循环。
- 优点：写代码灵活、调试方便
- 缺点：
  1. 不能脱离Python跑（C++/手机/边缘芯片不认.py）
  2. 每次推理重复搭计算逻辑，速度慢
  3. 难以做算子融合、量化等底层加速优化

#### 什么是「静态导出」？
提前把**完整计算流程 + 权重参数**打包成独立文件，不再依赖Python源码，直接交给推理引擎运行。
三者本质是三种不同标准的「模型菜谱」，用途、受众完全不同。

#### 1. torch.jit（TorchScript）：PyTorch老式私有菜谱
#### 通俗比喻
仅PyTorch自家C++/TorchMobile能看懂的内部菜谱，外部AI加速工具（TensorRT/NCNN等）不识别。

#### 两种生成方式
1. `torch.jit.trace` 追踪：喂一份样例输入，记录本次执行路径；if分支只会保存当前走的一条，动态分支失效
2. `torch.jit.script` 脚本解析：读取完整模型代码，保留全部if/循环逻辑，支持数据依赖分支

#### 产出文件
`.pt` TorchScript二进制文件

#### 优缺点
- ✅ 自家算子对齐精准，支持多自定义导出函数、训练图、张量数值判断分支
- ❌ 私有格式跨框架无法使用；官方停止新增功能，逐步淘汰

#### 适用场景
老旧PyTorch1.x项目；需要C++调用多个模型自定义接口；存在图片内容决定分支的逻辑。

#### 2. torch.export（ExportedProgram）：新一代统一标准草稿纸
#### 通俗比喻
PyTorch2.0+全新标准中间草稿，不直接拿来推理，专门统一整理模型逻辑，再转成TorchScript或ONNX。

#### 核心特性
1. 原生支持动态输入尺寸（可变batch、可变图片分辨率），解决jit大量动态shape坑
2. 强校验，导出阶段提前报错，不会部署到C++才崩溃
3. 输出干净标准化FX图，算子统一拆解，转ONNX稳定性大幅提升
4. 一份ExportedProgram可分流两条部署路线
   - 路线1：转TorchScript .pt 供LibTorch使用
   - 路线2：转ONNX通用模型供全平台加速

#### 硬性限制（关键）
不支持**数据依赖分支**：禁止`if 图片张量均值>0.5`这类靠输入内容改变计算流程的代码；
所有分支必须用`torch.where`算子抹平，不能保留原生if判断。

#### 适用场景
PyTorch2+新项目；仅推理场景；需要导出ONNX做跨平台部署；追求导出稳定性与动态尺寸支持。

#### 3. torch.onnx：全行业通用标准菜谱
#### 通俗比喻
全球AI框架统一通用菜谱，英伟达GPU、手机NCNN、开发板RKNN、Intel OpenVINO全部兼容读取。

#### 底层转换链路
- PyTorch1.x旧链路：Eager模型 → torch.jit → 翻译为ONNX
- PyTorch2.2+推荐链路：Eager模型 → torch.export草稿 → 翻译为ONNX，bug更少、动态形状支持更好

#### 产出文件
`.onnx` 通用二进制模型文件

#### 优缺点
- ✅ 跨平台、跨芯片通用，工业上线部署主流选择
- ❌ 算子翻译存在损耗，少量PyTorch独有算子无法映射标准ONNX算子，易精度下降、导出失败

#### 适用场景
需要部署到GPU/移动端/边缘开发板；使用ONNXRuntime、TensorRT、NCNN、RKNN等第三方推理引擎。

#### 三者核心对比简表
| 工具 | 产物 | 目标使用方 | 定位 |
|------|------|------------|------|
| torch.jit | .pt 私有TorchScript | 仅LibTorch/TorchMobile | 老式私有静态图，逐步淘汰 |
| torch.export | ExportedProgram中间图 | 中转草稿，不直接推理 | 新版统一标准导出入口 |
| torch.onnx | .onnx 通用模型 | 全平台所有AI推理框架 | 跨芯片跨框架部署标准 |

#### 两代导出流水线
#### 淘汰老流水线（PyTorch1.x）
Python动态模型 → torch.jit 生成.pt → 再翻译为.onnx
缺陷：jit追踪缺陷多，动态尺寸、分支极易导出失败

#### 官方推荐新流水线（PyTorch2+）
Python动态模型 → torch.export 标准化草稿
├─ 分支1：草稿 → 转换TorchScript .pt（C++ LibTorch推理）
└─ 分支2：草稿 → 转换ONNX .onnx（跨芯片/移动端部署）

#### 落地选型直白建议
1. 新项目、PyTorch≥2.2、仅推理、跨平台部署：优先 torch.export + torch.onnx
2. 仅用C++ LibTorch推理：torch.export 转TorchScript，不直接使用原生jit
3. 模型存在张量数值控制if分支、多导出函数、训练图导出：只能使用torch.jit.script
4. 老旧1.x存量项目：保留torch.jit不做迁移

#### 一句话总结
- torch.jit：过时PyTorch私有模型格式；
- torch.export：新版统一标准化导出中间工具，官方主推；
- torch.onnx：全行业通用上线部署模型格式。

In [ ]:
# 1. torch.jit.trace 基础示例
import torch

class DemoNet(torch.nn.Module):
    def forward(self, x):
        return x + torch.ones_like(x)

model = DemoNet().eval()
inp = torch.randn(1, 3, 224, 224)
# trace追踪导出
traced_mod = torch.jit.trace(model, inp)
traced_mod.save("jit_trace_model.pt")

In [ ]:
# 2. torch.export 标准导出 + 转TorchScript示例
import torch

class DemoNet(torch.nn.Module):
    def forward(self, x):
        return x + torch.ones_like(x)

model = DemoNet().eval()
inp_args = (torch.randn(1, 3, 224, 224),)
# 生成标准化ExportedProgram
ep = torch.export.export(model, inp_args)
# 转换为TorchScript文件
script_mod = ep.module()
script_mod.save("export_ts_model.pt")

In [ ]:
# 3. torch.export 链路导出ONNX（推荐稳定写法）
import torch

class DemoNet(torch.nn.Module):
    def forward(self, x):
        return x + torch.ones_like(x)

model = DemoNet().eval()
inp = torch.randn(1, 3, 224, 224)

torch.onnx.export(
    model,
    inp,
    "export_based_model.onnx",
    opset_version=18,
    dynamic_axes={"x": {0: "batch_size"}}  # 动态batch维度
)